# Pendulum core example

We will run the pendulum in its own simulation thread and send torque commands from a simple scan loop in this notebook. Select the Python 3.13 environment belonging to `bs-simulators` as the notebook kernel.


In [295]:
import time

from bs_simulators.runtime import PendulumSimulation
import numpy as np
import pandas as pd

## Create and inspect the simulator

We will choose an initial angle of `pi - 0.1` rad and a physics interval `dt` of `0.01` s. The controller identity is `python`. Creating the object does not start its simulation thread.


In [296]:
simulation = PendulumSimulation(theta0=np.pi - 0.25, dt=0.01, active_controller="python")
simulation.snapshot()

{'t': 0.0,
 'x': [2.891592653589793, 0.0],
 'u': [0.0],
 'seq': 0,
 'input_seq': 0,
 'pending_input_seq': None,
 'applied_at_sim_time': 0.0,
 'wall_elapsed': 0.0,
 'lag': 0.0,
 'status': 'ready',
 'parameters': {'m': 1.0, 'L': 1.0, 'drag': 0.1},
 'active_controller': 'python',
 'controller_id': None,
 'received_input_seq': 0,
 'received_at_monotonic': None,
 'source_timestamp': None,
 'received_source_timestamp': None,
 'applied_received_at_monotonic': None}

`parameters` contains mass `m`, length `L`, and viscous drag. The following cells expose the signal descriptions and the initial measurements as ordinary Python objects.


In [297]:
pd.DataFrame(simulation.describe_signals())

,name,unit,dtype,direction,lower,upper
0,theta,rad,float64,output,NaN,None
1,omega,rad/s,float64,output,NaN,None
2,torque,N m,float64,input,NaN,None
3,time,s,float64,output,0.0,None
4,status,,string,output,NaN,None


In [298]:
simulation.read_measurements()

{'theta': 2.891592653589793,
 'omega': 0.0,
 'torque': 0.0,
 'time': 0.0,
 'status': 'ready'}

The writable signal is `torque`, in N m. Measurements include `theta` in radians and `omega` in rad/s. We can also inspect a public method directly from the notebook.


In [299]:
help(PendulumSimulation.start)

Help on function start in module bs_simulators.runtime.core:

start(self, *, duration=None, record=None)
    Start a worker; optionally record initial state and each committed step.

    ``duration`` is in simulated seconds. It must be a whole number of
    fixed steps so recorded time and requested duration agree exactly.
    ``record`` is a JSONL path, created exclusively to avoid overwrites.



## HTTP viewer

Run the next cell to serve a browser viewer of the same `simulation` object. The operating system selects an available local port. The HTTP thread serves observations while the notebook remains the selected controller (`active_controller="python"`).

Open the displayed link **before running the scan loop**. The viewer initially shows the initial state; it animates when the next section starts the simulation. You can skip this section for the core-only example.


In [300]:
import threading

from bs_simulators.comms.http import create_server

server = create_server(simulation, port=0)  # port=0 means the OS selects an available port
server_thread = threading.Thread(target=server.serve_forever, daemon=True)
server_thread.start()
viewer_url = f"http://127.0.0.1:{server.server_address[1]}/"
f"Open the pendulum viewer: {viewer_url}"

'Open the pendulum viewer: http://127.0.0.1:55906/'

## Run a scan loop

We will use a PD controller with `pi` radians as its reference. Each scan reads measurements, calculates torque, writes the input vector, and waits `0.05` s. The simulation thread integrates independently at `dt=0.01` s and holds the last input between commands.

The loop runs 100 scans, approximately five seconds. `set_input([torque])` is the direct Python convenience method for the selected controller.

In [ ]:
scan_period = 0.05
kp, kd = 20.0, 4.0
ref = np.pi

try:
    e_prev = 0.0
    simulation.start()
    for _ in range(100):
        # Read measurements
        measurements = simulation.read_measurements()

        # Compute the PD control law
        e = ref - measurements["theta"]
        de_dt = (e - e_prev) / scan_period # Forward Euler approximation
        e_prev = e
        torque = kp * e + kd * de_dt

        # Update the plant
        simulation.set_input([torque])

        # Wait for the next scan period
        print(f"theta={measurements['theta']:.3f} rad, torque={torque:.3f} N·m", end="\r")
        time.sleep(scan_period)
        
finally:
    simulation.stop()

`kp` acts on the angular error and `kd` damps angular velocity. The sleep is an approximate scan delay; the total scan period also includes the execution time of the loop body.

After stopping, the object remains available for inspection. To repeat the experiment, create a new `PendulumSimulation` by running the setup cells again.


In [302]:
simulation.read_measurements()

{'theta': 3.1415926506702734,
 'omega': -8.727075871565453e-09,
 'torque': 2.296855239336537e-07,
 'time': 6.359999999999909,
 'status': 'stopped'}

The full snapshot also exposes simulated time `t`, state vector `x=[theta, omega]`, applied input `u=[torque]`, and command sequences. An accepted command is applied at an integration boundary; `input_seq` identifies the command used by the latest committed state.


In [303]:
simulation.snapshot()

{'t': 6.359999999999909,
 'x': [3.1415926506702734, -8.727075871565453e-09],
 'u': [2.296855239336537e-07],
 'seq': 636,
 'input_seq': 100,
 'pending_input_seq': None,
 'applied_at_sim_time': 6.28999999999991,
 'wall_elapsed': 6.375786300050095,
 'lag': 0.015786300050185886,
 'status': 'stopped',
 'parameters': {'m': 1.0, 'L': 1.0, 'drag': 0.1},
 'active_controller': 'python',
 'controller_id': 'python',
 'received_input_seq': 100,
 'received_at_monotonic': 1267391.4210367,
 'source_timestamp': None,
 'received_source_timestamp': None,
 'applied_received_at_monotonic': 1267391.4210367}

## Close the HTTP viewer server

If you started the optional HTTP viewer, run this cell when you finish inspecting the final state. The server remains available after the control loop stops, so the browser can still display that state.

Close the server before repeating the setup cells: it holds a reference to the simulation object passed to `create_server()`.


In [304]:
server.shutdown()
server.server_close()
server_thread.join()